---
## CELL 1 — Install libraries

In [ ]:
# ── CELL 1: Install all required libraries ────────────────────────────────
!pip install -q sentence-transformers transformers torch spacy scikit-learn scipy datasets
!python -m spacy download en_core_web_sm -q
print('✅ All libraries installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 36.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ All libraries installed


---
## CELL 2 — Load models

In [ ]:
# ── CELL 2: Load all NLP models ───────────────────────────────────────────
# This cell downloads models on first run (~3 mins). Cached after that.

from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import spacy, numpy as np, json, warnings
warnings.filterwarnings('ignore')

print('Loading Sentence-BERT...')
sbert = SentenceTransformer('all-MiniLM-L6-v2')

print('Loading NLI model...')
nli = pipeline(
    'text-classification',
    model='cross-encoder/nli-deberta-v3-small',
    device=0 if __import__('torch').cuda.is_available() else -1
)

print('Loading spaCy...')
nlp = spacy.load('en_core_web_sm')

import torch
device = 'GPU ✅' if torch.cuda.is_available() else 'CPU (consider enabling GPU in Runtime > Change runtime type)'
print(f'\n✅ All models loaded | Device: {device}')

Loading Sentence-BERT...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Loading spaCy...

✅ All models loaded | Device: GPU ✅


---
## CELL 3 — Define all 6 metric functions

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FINAL IMPROVED CELL 3 — Replace your current Cell 3 with this
# Key change: P (contradiction) completely removed from scoring
# Replaced with: Internal Consistency check (self-contradiction only)
# ═══════════════════════════════════════════════════════════════════

from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import spacy, numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── R: Relevance ──────────────────────────────────────────────────
def relevance_score(question, student_answer):
    """Semantic alignment of full answer to the question."""
    q_emb = sbert.encode(question,       convert_to_tensor=True)
    a_emb = sbert.encode(student_answer, convert_to_tensor=True)
    return round(float(util.cos_sim(q_emb, a_emb).clamp(0, 1)), 4)


# ── V: Coverage ───────────────────────────────────────────────────
def coverage_score(student_answer, rubric_points):
    """
    Sentence-level matching — finds best sentence per rubric point.
    4-tier partial scoring. Lower threshold = fairer for long answers.
    """
    doc   = nlp(student_answer)
    sents = [s.text.strip() for s in doc.sents if len(s.text.strip()) > 10]
    if not sents:
        sents = [student_answer]

    per_point = []
    for point in rubric_points:
        p_emb    = sbert.encode(point, convert_to_tensor=True)
        best_sim = 0.0
        for sent in sents:
            s_emb = sbert.encode(sent, convert_to_tensor=True)
            sim   = float(util.cos_sim(s_emb, p_emb))
            if sim > best_sim:
                best_sim = sim

        if   best_sim >= 0.65: partial = 1.0
        elif best_sim >= 0.50: partial = 0.75
        elif best_sim >= 0.40: partial = 0.50
        elif best_sim >= 0.30: partial = 0.25
        else:                  partial = 0.0

        per_point.append({
            'rubric_point': point,
            'similarity':   round(best_sim, 4),
            'partial':      partial,
            'covered':      best_sim >= 0.40
        })

    score = sum(p['partial'] for p in per_point) / len(rubric_points)
    return {
        'score':         round(score, 4),
        'per_point':     per_point,
        'covered_count': sum(1 for p in per_point if p['covered']),
        'total_points':  len(rubric_points)
    }


# ── C: Consistency ────────────────────────────────────────────────
def consistency_score(student_answer, rubric_points):
    """
    Sentence-level NLI — finds most relevant sentence per rubric point
    before running NLI. Skips irrelevant sentences entirely.
    """
    doc   = nlp(student_answer)
    sents = [s.text.strip() for s in doc.sents if len(s.text.strip()) > 10]
    if not sents:
        sents = [student_answer[:200]]

    scores = []
    for point in rubric_points:
        p_emb     = sbert.encode(point, convert_to_tensor=True)
        best_sent = student_answer[:300]
        best_sim  = 0.0
        for sent in sents:
            s_emb = sbert.encode(sent, convert_to_tensor=True)
            sim   = float(util.cos_sim(s_emb, p_emb))
            if sim > best_sim:
                best_sim  = sim
                best_sent = sent

        if best_sim < 0.25:
            scores.append(0.5)
            continue

        res   = nli(f'{point} [SEP] {best_sent[:300]}')[0]
        label = res['label'].lower()
        conf  = res['score']

        if   'entail'  in label: scores.append(1.0 * conf)
        elif 'neutral' in label: scores.append(0.5)
        else:                    scores.append(0.3)

    return round(float(np.mean(scores)), 4)


# ── Q: Reasoning Quality ──────────────────────────────────────────
def reasoning_quality_score(student_answer):
    """
    DEFINITIVE v4 — measures explanation depth, not just markers.

    Three signals combined:
    1. Explicit discourse markers (because, therefore, etc.)
    2. Average sentence length — longer = more explanation
    3. Vocabulary richness — varied words = richer explanation

    This correctly handles:
    - Good answers without "because/therefore" → length + vocab rescues them
    - Keyword-only poor answers → short sentences + low vocab pulls Q down
    """
    explicit_markers = [
        'because', 'therefore', 'thus', 'hence', 'since',
        'as a result', 'consequently', 'this means', 'which causes',
        'leading to', 'due to', 'this leads', 'resulting in', 'so that',
        'this causes', 'which results', 'this results', 'in order to',
        'which allows', 'this allows', 'this ensures', 'which ensures',
        'this prevents', 'which prevents', 'this improves', 'enabling',
        'which enables', 'this enables', 'which reduces', 'this reduces',
        'which increases', 'which means', 'this guarantees', 'thereby',
        'this avoids', 'accordingly', 'for this reason', 'as such',
        'in turn', 'which in turn', 'and therefore', 'and thus'
    ]

    doc        = nlp(student_answer)
    sents      = [s for s in doc.sents if len(s.text.strip()) > 3]
    words      = [t.text.lower() for t in doc
                  if t.is_alpha and not t.is_stop and len(t.text) > 2]

    if not sents or not words:
        return 0.0

    # Signal 1: discourse markers (0–1)
    text_lower   = student_answer.lower()
    found        = sum(1 for m in explicit_markers if m in text_lower)
    marker_score = min(found / 3.0, 1.0)   # 3+ markers = full score

    # Signal 2: average words per sentence (0–1)
    # Poor answer: avg 5-7 words/sent  → low score
    # Good answer: avg 12-20 words/sent → high score
    avg_sent_len  = np.mean([len(s.text.split()) for s in sents])
    length_score  = min((avg_sent_len - 5) / 15.0, 1.0)   # 20 words = full
    length_score  = max(0.0, length_score)

    # Signal 3: vocabulary richness — unique content words / total
    # Poor answer: repeats same few words → low richness
    # Good answer: varied technical vocabulary → high richness
    unique_words   = len(set(words))
    total_words    = len(words)
    vocab_richness = min(unique_words / max(total_words, 1) * 1.5, 1.0)

    # Signal 4: dependency-based causal detection
    causal_deps = sum(1 for token in doc
                      if token.dep_ in ('advcl','relcl','csubj','xcomp')
                      and token.head.pos_ in ('VERB','AUX'))
    dep_score   = min(causal_deps / max(len(sents), 1), 1.0)

    # Weighted combination
    combined = (0.30 * marker_score  +
                0.30 * length_score  +
                0.25 * vocab_richness +
                0.15 * dep_score)

    return round(min(combined, 1.0), 4)



# ── H: Coherence ─────────────────────────────────────────────────
def coherence_score(student_answer):
    """Logical flow between consecutive sentences."""
    doc   = nlp(student_answer)
    sents = [s.text.strip() for s in doc.sents if len(s.text.strip()) > 5]
    if len(sents) < 2:
        return 0.5
    embs = sbert.encode(sents, convert_to_tensor=True)
    sims = [float(util.cos_sim(embs[i], embs[i+1]))
            for i in range(len(embs)-1)]
    return round(float(np.mean(sims)), 4)


# ── : Internal Consistency (replaces contradiction penalty) ──────
def internal_consistency_score(student_answer):
    """
    NEW: Checks if the student contradicts THEMSELVES within
    their own answer — not against rubric.

    This is a genuine quality signal:
      - Student says X in sentence 2 and NOT X in sentence 5 → low score
      - Student is consistent throughout → high score

    Uses NLI between sentence pairs within the answer only.
    Returns 0.0–1.0 where 1.0 = fully self-consistent.
    """
    doc   = nlp(student_answer)
    sents = [s.text.strip() for s in doc.sents if len(s.text.strip()) > 15]

    # Need at least 2 sentences to check internal consistency
    if len(sents) < 2:
        return 1.0   # single sentence = trivially consistent

    # Check consecutive sentence pairs (not all pairs — too slow)
    # Compare each sentence against the next one
    consistency_scores = []
    pairs_to_check = min(len(sents) - 1, 4)   # max 4 pairs to keep fast

    for i in range(pairs_to_check):
        s1 = sents[i][:200]
        s2 = sents[i + 1][:200]
        res   = nli(f'{s1} [SEP] {s2}')[0]
        label = res['label'].lower()
        conf  = res['score']

        if   'entail'     in label: consistency_scores.append(1.0)
        elif 'neutral'    in label: consistency_scores.append(0.8)
        else:                       consistency_scores.append(0.4)

    return round(float(np.mean(consistency_scores)), 4)


# ── Master evaluate function ──────────────────────────────────────
def evaluate_answer(question, student_answer, rubric_points,
                    max_marks=10.0, weights=None):
    """
    FINAL formula with two interaction terms:

    1. V × Q interaction (existing):
       effective_V = V * (0.5 + 0.5*Q)
       → coverage without explanation = half credit

    2. NEW: Answer length gate
       Very short answers (< 50 words) get a length_multiplier < 1.0
       This directly captures the keyword-dumping pattern:
         "Deadlock is when process waits. Conditions are four." → 10 words
         → length_multiplier = 0.40 → score capped at ~4/10 max

       Long well-explained answers (> 120 words) get full credit:
         → length_multiplier = 1.0

    3. NEW: Sentence completeness check
       Counts fragment sentences (< 5 words) as a quality penalty
       "Mutual exclusion. Hold wait. No preemption." = 3 fragments
       → fragment_penalty reduces score proportionally
    """
    if weights is None:
        weights = {
            'R':  0.15,
            'V':  0.25,
            'C':  0.15,
            'Q':  0.30,
            'H':  0.10,
            'IC': 0.05
        }

    R   = relevance_score(question, student_answer)
    cov = coverage_score(student_answer, rubric_points)
    V   = cov['score']
    C   = consistency_score(student_answer, rubric_points)
    Q   = reasoning_quality_score(student_answer)
    H   = coherence_score(student_answer)
    IC  = internal_consistency_score(student_answer)

    # ── Interaction 1: V×Q (coverage needs explanation) ──────────
    effective_V = V * (0.5 + 0.5 * Q)

    raw   = (weights.get('R',  0.15) * R           +
             weights.get('V',  0.25) * effective_V  +
             weights.get('C',  0.15) * C            +
             weights.get('Q',  0.30) * Q            +
             weights.get('H',  0.10) * H            +
             weights.get('IC', 0.05) * IC)

    final = round(max(0.0, min(raw, 1.0)) * max_marks, 2)

    return {
        'R_relevance':         R,
        'V_coverage':          V,
        'V_effective':         round(effective_V, 4),
        'C_consistency':       C,
        'Q_reasoning':         Q,
        'H_coherence':         H,
        'IC_self_consistency': IC,
        'P_penalty':           0.0,
        'word_count':          len(student_answer.split()),
        'length_multiplier':   1.0,   # always 1 now
        'fragment_penalty':    1.0,   # always 1 now
        'final_score':         final,
        'max_marks':           max_marks,
        'coverage_detail':     cov['per_point'],
        'covered_count':       cov['covered_count'],
        'total_points':        cov['total_points']
    }

---
## CELL 4 — Quick test on 2 sample answers

In [ ]:
# ── CELL 4: Quick sanity test ─────────────────────────────────────────────
# Tests a good answer and a poor answer on the same question.
# Good answer should score much higher than poor answer.

test_question = 'Explain deadlock and its four necessary conditions.'
test_rubric   = [
    'deadlock is a situation where processes wait indefinitely for resources',
    'mutual exclusion means only one process can use a resource at a time',
    'hold and wait means a process holds resources while waiting for more',
    'no preemption means resources cannot be forcibly removed',
    'circular wait means a chain of processes each waiting for the next'
]

good_answer = """
Deadlock is a state in OS where processes are permanently blocked because each
is waiting for a resource held by another. Mutual exclusion ensures only one
process uses a resource at a time. Hold and wait means a process holds one
resource while waiting for more. Since no preemption is allowed, resources
cannot be taken away forcibly. Circular wait creates a cycle where P1 waits
for P2, P2 waits for P3, and P3 waits for P1, therefore forming a deadlock.
"""

poor_answer = 'Deadlock is a problem in OS. It has four conditions.'

print('Testing good answer...')
r_good = evaluate_answer(test_question, good_answer, test_rubric, max_marks=10)

print('Testing poor answer...')
r_poor = evaluate_answer(test_question, poor_answer, test_rubric, max_marks=10)

print('\n' + '='*55)
print(f'{"Metric":<22} {"Good answer":>14} {"Poor answer":>14}')
print('='*55)
metrics = ['R_relevance','V_coverage','C_consistency',
           'Q_reasoning','H_coherence','P_penalty','final_score']
for m in metrics:
    print(f'{m:<22} {r_good[m]:>14} {r_poor[m]:>14}')
print('='*55)
print(f'\n✅ System working correctly')
print(f'   Good answer: {r_good["final_score"]} / 10')
print(f'   Poor answer: {r_poor["final_score"]} / 10')

Testing good answer...
Testing poor answer...

Metric                    Good answer    Poor answer
R_relevance                    0.7158         0.7437
V_coverage                        1.0            0.3
C_consistency                  0.6572            0.5
Q_reasoning                      0.89           0.25
H_coherence                    0.3196         0.0488
P_penalty                         0.0            0.0
final_score                      7.71           3.53

✅ System working correctly
   Good answer: 7.71 / 10
   Poor answer: 3.53 / 10


---
## CELL 5 — Load dataset (50 student answers)

In [ ]:
# ── CELL 5: Create the 50-answer dataset inline ───────────────────────────
# No file upload needed — dataset is embedded directly here

DATASET = {
  'questions': [
    {
      'id': 'Q1',
      'question': 'Explain deadlock and its four necessary conditions.',
      'max_marks': 10,
      'rubric_points': [
        'deadlock is a situation where processes wait indefinitely for resources held by each other',
        'mutual exclusion means only one process can use a resource at a time',
        'hold and wait means a process holds at least one resource while waiting for additional resources',
        'no preemption means resources cannot be forcibly taken from a process',
        'circular wait means a chain of processes each waiting for a resource held by the next'
      ],
      'student_answers': [
        {'id':'Q1_A1','answer':'Deadlock is a situation in operating systems where two or more processes are unable to proceed because each is waiting for a resource held by another. There are four necessary conditions for deadlock. First, mutual exclusion means a resource can only be held by one process at a time. Second, hold and wait means a process is holding at least one resource and waiting to acquire additional resources. Third, no preemption means resources cannot be forcibly removed from a process holding them. Fourth, circular wait means there exists a circular chain of processes where each process waits for a resource held by the next process in the chain. All four conditions must hold simultaneously for deadlock to occur.','avg_human_score':9.75,'quality':'excellent'},
        {'id':'Q1_A2','answer':'Deadlock occurs when processes are stuck waiting for each other. The four conditions are mutual exclusion where resources cannot be shared, hold and wait where processes keep resources while asking for more, no preemption which means you cannot take resources away from a process, and circular wait which means processes form a cycle waiting for each other. Because these four conditions exist together, deadlock cannot be resolved without external intervention.','avg_human_score':7.75,'quality':'good'},
        {'id':'Q1_A3','answer':'Deadlock is when processes wait forever. Conditions are mutual exclusion, hold and wait, no preemption, circular wait. Mutual exclusion means one process uses resource. Hold and wait is when process holds resource. No preemption means cannot take resource. Circular wait is a cycle.','avg_human_score':5.25,'quality':'average'},
        {'id':'Q1_A4','answer':'Deadlock happens in OS. There are four conditions for deadlock to happen. Mutual exclusion is one condition. The other conditions are hold and wait, no preemption, and circular wait. These conditions are important in operating systems.','avg_human_score':3.25,'quality':'poor'},
        {'id':'Q1_A5','answer':'Deadlock is a critical problem in concurrent systems where multiple processes compete for limited resources. It occurs when each process in a set is waiting for an event that only another process in the set can cause. Mutual exclusion ensures resources are non-shareable at any given time. Hold and wait allows processes to hold allocated resources while simultaneously requesting new ones, therefore creating resource contention. No preemption guarantees that resources are released voluntarily, since forceful removal could corrupt process state. Circular wait creates a cycle in the resource allocation graph, leading to an indefinite wait.','avg_human_score':9.25,'quality':'excellent'},
        {'id':'Q1_A6','answer':'Deadlock means no process can continue. Mutual exclusion is needed. Processes hold and wait. Resources cannot be preempted. There is a circular chain. So deadlock occurs when all four conditions are present at the same time.','avg_human_score':4.0,'quality':'below_average'},
        {'id':'Q1_A7','answer':'Deadlock is a condition where a set of processes are blocked because each process is holding a resource and waiting for another resource acquired by some other process. The first condition is mutual exclusion which means at least one resource must be held in a non-shareable mode. The second is hold and wait. The third condition called no preemption says that a resource can be released only voluntarily by the process. The fourth is circular wait. Therefore all four conditions are necessary for deadlock.','avg_human_score':7.0,'quality':'good'},
        {'id':'Q1_A8','answer':'Deadlock is problem in OS where processes get stuck. Four conditions exist. These are called Coffman conditions. They are mutual exclusion hold and wait no preemption circular wait. Without all four conditions deadlock will not happen.','avg_human_score':2.75,'quality':'poor'},
        {'id':'Q1_A9','answer':'Deadlock is a state where processes are permanently blocked. Mutual exclusion condition states that resources involved must be unshareable. Hold and wait condition means that a process must be holding one resource and waiting for others. Since no preemption is allowed, resources held by a process can only be released by that process voluntarily. Circular wait means process P1 waits for P2, P2 waits for P3, and P3 waits back for P1, creating a cycle. Because all these conditions are necessary, removing even one prevents deadlock.','avg_human_score':8.25,'quality':'good'},
        {'id':'Q1_A10','answer':'Deadlock is when process waits. OS has deadlock problem. Conditions are four. Mutual exclusion. Hold wait. No preemption. Circular. These are conditions.','avg_human_score':1.75,'quality':'very_poor'}
      ]
    },
    {
      'id': 'Q2',
      'question': 'Explain CPU scheduling and compare FCFS and Round Robin algorithms.',
      'max_marks': 10,
      'rubric_points': [
        'CPU scheduling determines which process runs next on the CPU from the ready queue',
        'FCFS schedules processes in order of arrival and is non-preemptive',
        'FCFS causes convoy effect where short processes wait behind long processes',
        'Round Robin assigns a fixed time quantum to each process in cyclic order',
        'Round Robin is preemptive and provides fair CPU allocation among all processes'
      ],
      'student_answers': [
        {'id':'Q2_A1','answer':'CPU scheduling is the process by which the operating system decides which process in the ready queue gets to use the CPU next. FCFS or First Come First Served schedules processes strictly in the order they arrive. It is non-preemptive, meaning once a process starts executing it runs to completion. The main disadvantage of FCFS is the convoy effect, where short processes are forced to wait behind long processes, thus increasing average waiting time significantly. Round Robin is a preemptive algorithm that assigns a fixed time quantum to each process. After the quantum expires, the process is moved to the back of the ready queue and the next process runs. This ensures fair CPU allocation because every process gets an equal share of CPU time.','avg_human_score':9.75,'quality':'excellent'},
        {'id':'Q2_A2','answer':'CPU scheduling selects which process runs next. FCFS runs processes in arrival order and is non-preemptive. It has convoy effect problem where long processes block short ones. Round Robin gives each process a time slice called quantum and preempts after quantum expires. Round Robin is fairer than FCFS because all processes get equal CPU time. Therefore Round Robin is preferred for time-sharing systems.','avg_human_score':7.75,'quality':'good'},
        {'id':'Q2_A3','answer':'CPU scheduling is important in OS. FCFS is first come first served. Round Robin uses time quantum. FCFS is non preemptive. Round Robin is preemptive. FCFS has convoy effect. Round Robin is better for interactive systems.','avg_human_score':4.75,'quality':'below_average'},
        {'id':'Q2_A4','answer':'CPU scheduling decides process execution order. FCFS means processes execute in the order they arrive in the ready queue. Since it is non-preemptive, a long process can cause shorter processes to wait for a long time, which is called the convoy effect. Round Robin solves this by using a time quantum, after which the running process is preempted and placed at the end of the queue. As a result, Round Robin provides better response time and is suitable for interactive environments.','avg_human_score':8.75,'quality':'excellent'},
        {'id':'Q2_A5','answer':'Scheduling is done by CPU. FCFS and Round Robin are two algorithms. FCFS is non preemptive. Round Robin is preemptive. Both are used in operating systems.','avg_human_score':2.25,'quality':'poor'},
        {'id':'Q2_A6','answer':'CPU scheduling manages process execution on CPU. FCFS executes processes as they arrive therefore it is simple to implement. However since FCFS is non-preemptive long processes block shorter ones causing convoy effect which increases average waiting time. Round Robin addresses this by preempting processes after a fixed time quantum hence no single process starves others. The choice of time quantum in Round Robin is critical because a large quantum makes it behave like FCFS while a very small quantum increases context switch overhead.','avg_human_score':9.0,'quality':'excellent'},
        {'id':'Q2_A7','answer':'CPU scheduling is selection of process from ready queue. FCFS is first algorithm. Round Robin is second algorithm. FCFS has problems. Round Robin is better. Time quantum is used in Round Robin.','avg_human_score':3.0,'quality':'poor'},
        {'id':'Q2_A8','answer':'CPU scheduling allocates processor time to processes. First Come First Served is the simplest scheduling algorithm where processes are served in arrival order. It is non-preemptive so once a process occupies the CPU it continues until it finishes or blocks. This causes convoy effect where CPU-bound long processes make I/O-bound short processes wait. Round Robin improves on this by introducing preemption via time quantum. Each process runs for at most one quantum before being switched out.','avg_human_score':8.25,'quality':'good'},
        {'id':'Q2_A9','answer':'Scheduling is when OS picks process. FCFS and Round Robin both schedule. FCFS comes first. Round Robin has quantum. They are different algorithms in OS.','avg_human_score':1.75,'quality':'very_poor'},
        {'id':'Q2_A10','answer':'CPU scheduling determines order of process execution in the ready queue. FCFS is non-preemptive and executes processes in arrival order leading to convoy effect when long processes arrive first. Round Robin is preemptive and assigns equal time quantum to each process in cyclic manner. Because of preemption Round Robin prevents any process from monopolising CPU. However Round Robin requires careful selection of quantum size since very small quantum causes excessive context switches while very large quantum degrades to FCFS behaviour.','avg_human_score':8.25,'quality':'good'}
      ]
    },
    {
      'id': 'Q3',
      'question': 'Explain binary search trees and their operations.',
      'max_marks': 10,
      'rubric_points': [
        'a binary search tree is a binary tree where left child is smaller and right child is greater than the parent node',
        'search operation compares target with current node and traverses left or right accordingly',
        'insertion places new node at correct position maintaining BST property',
        'deletion handles three cases leaf node node with one child node with two children',
        'BST operations have average time complexity of O log n but degrade to O n in worst case for skewed trees'
      ],
      'student_answers': [
        {'id':'Q3_A1','answer':'A binary search tree is a binary tree with a special ordering property where for every node, all values in the left subtree are smaller and all values in the right subtree are greater. Search starts at root and compares target value with current node, going left if smaller and right if larger, until the value is found or a null pointer is reached. Insertion follows the same path as search and places the new node at the appropriate leaf position to maintain the BST property. Deletion is the most complex operation with three cases: if the node is a leaf it is simply removed, if it has one child that child replaces it, and if it has two children the in-order successor replaces it. Average time complexity for all operations is O(log n) since each comparison eliminates half the tree, however a skewed tree degrades to O(n) in the worst case.','avg_human_score':9.75,'quality':'excellent'},
        {'id':'Q3_A2','answer':'BST is a tree where left node is smaller and right node is larger than parent. Search compares value with node and goes left or right. Insertion adds node at correct position. Deletion removes node with three cases for leaf, one child, two children. Time complexity is O(log n) average and O(n) worst case for skewed trees.','avg_human_score':7.25,'quality':'good'},
        {'id':'Q3_A3','answer':'Binary search tree is a tree data structure. Left side has smaller values and right side has larger values. We can search insert and delete in BST. It is efficient data structure used in many applications.','avg_human_score':3.25,'quality':'poor'},
        {'id':'Q3_A4','answer':'A BST maintains sorted order where left subtree contains values less than root and right subtree contains values greater than root. For search, we compare the key with current node and recursively search left subtree if key is smaller or right subtree if key is larger, therefore achieving efficient lookup. Insertion follows the search path and creates a new leaf node at the correct position thus maintaining BST property. Deletion has three cases. Deleting a leaf is straightforward. Deleting a node with one child replaces the node with its child. Deleting a node with two children replaces it with inorder successor.','avg_human_score':9.25,'quality':'excellent'},
        {'id':'Q3_A5','answer':'BST has left smaller right greater property. Search traverses tree. Insert adds node. Delete removes node. Complexity is O log n.','avg_human_score':2.25,'quality':'very_poor'},
        {'id':'Q3_A6','answer':'Binary search tree is ordered binary tree. Left child is always less than parent and right child is always greater. Search operation starts at root and moves left or right based on comparison hence it is efficient. Insertion maintains BST ordering property by finding correct position. Deletion is complex because we need to handle three different cases to maintain BST property after removal. Time complexity is O log n on average but becomes O n for unbalanced trees because the tree degenerates into a linked list.','avg_human_score':7.75,'quality':'good'},
        {'id':'Q3_A7','answer':'BST is data structure. It is a tree. We can do operations on BST. Search and insert and delete are operations. It is O log n.','avg_human_score':1.75,'quality':'very_poor'},
        {'id':'Q3_A8','answer':'A binary search tree organises data so that left subtree has smaller keys and right subtree has larger keys than root. This property enables efficient search since at every node we eliminate half the remaining tree. Insertion places new element by following search path until empty spot is found. Deletion handles leaf nodes by direct removal, nodes with single child by bypassing the node, and nodes with two children by replacing with inorder successor to maintain BST property. Since each operation reduces search space by half on average, time complexity is O log n.','avg_human_score':8.75,'quality':'excellent'},
        {'id':'Q3_A9','answer':'Binary search tree has ordering property. Search is efficient. Insert and delete modify tree. Complexity depends on tree height. Balanced tree is better than unbalanced.','avg_human_score':3.25,'quality':'poor'},
        {'id':'Q3_A10','answer':'BST is a binary tree where for any node the left subtree contains only nodes with keys less than that node and right subtree contains only nodes with keys greater. Search operation works by comparing target with current node and recursively going left if target is smaller or right if larger. Insertion uses same logic as search to find correct position. Deletion is most complex since we must maintain BST property. Leaf deletion is simple, one child case replaces node with its child, two children case requires finding inorder successor. Average O log n complexity comes from halving search space at each level.','avg_human_score':8.75,'quality':'excellent'}
      ]
    },
    {
      'id': 'Q4',
      'question': 'Explain virtual memory and the concept of paging.',
      'max_marks': 10,
      'rubric_points': [
        'virtual memory allows processes to use more memory than physically available by using disk as extension of RAM',
        'paging divides logical memory into fixed size blocks called pages and physical memory into frames of same size',
        'page table maps virtual page numbers to physical frame numbers for address translation',
        'page fault occurs when a required page is not in physical memory and must be loaded from disk',
        'demand paging loads pages only when needed thus reducing memory usage and improving multiprogramming'
      ],
      'student_answers': [
        {'id':'Q4_A1','answer':'Virtual memory is a memory management technique that gives processes the illusion of having more memory than physically available by using disk space as an extension of RAM. Paging is the most common implementation of virtual memory. It divides logical address space into fixed-size units called pages and physical memory into equally sized frames. The page table maintains a mapping from virtual page numbers to physical frame numbers, enabling address translation by the memory management unit. When a process accesses a page not currently in physical memory, a page fault occurs, causing the OS to load the required page from disk into a free frame. Demand paging extends this by loading pages only when they are actually accessed rather than loading the entire process upfront.','avg_human_score':9.75,'quality':'excellent'},
        {'id':'Q4_A2','answer':'Virtual memory lets processes use more memory than available RAM by using disk. Paging divides memory into pages and frames of equal size. Page table maps pages to frames for address translation. When page is not in memory a page fault occurs and OS loads it from disk. Demand paging loads pages only when needed which saves memory.','avg_human_score':7.75,'quality':'good'},
        {'id':'Q4_A3','answer':'Virtual memory is a technique in OS. Paging is used to implement virtual memory. Pages and frames are used. Page table is maintained. Page fault happens when page is not found.','avg_human_score':3.25,'quality':'poor'},
        {'id':'Q4_A4','answer':'Virtual memory extends available memory by using secondary storage as backing store for pages not currently needed in RAM. Paging divides both logical and physical memory into equal fixed-size blocks called pages and frames respectively. The page table stored in memory maps each virtual page to its corresponding physical frame. When CPU generates a virtual address, MMU uses page table to translate it to physical address. If the page is not present in RAM, a page fault interrupt is generated therefore OS must fetch the page from disk. Demand paging improves efficiency by loading pages lazily only when referenced.','avg_human_score':9.25,'quality':'excellent'},
        {'id':'Q4_A5','answer':'Virtual memory uses disk as RAM. Paging divides memory. Page table is used. Page fault is error. Demand paging is efficient.','avg_human_score':1.75,'quality':'very_poor'},
        {'id':'Q4_A6','answer':'Virtual memory allows execution of programs that are larger than physical memory since only active portions need to be in RAM at any time. Paging implements this by breaking logical address space into pages and physical memory into frames of equal size. Page table provides the mapping needed for address translation from virtual to physical addresses. A page fault is triggered when the referenced page is absent from physical memory causing the OS to swap it in from disk. Demand paging is lazy loading strategy where pages are brought into memory only on demand hence reducing initial load time.','avg_human_score':8.75,'quality':'excellent'},
        {'id':'Q4_A7','answer':'Virtual memory is memory management. Paging breaks memory into pages. Frames are in physical memory. Page table maps pages. Page fault occurs sometimes.','avg_human_score':2.75,'quality':'poor'},
        {'id':'Q4_A8','answer':'Virtual memory provides abstraction allowing processes to have large address space independent of physical memory size. Paging divides logical address space into pages and physical memory into frames where page size equals frame size. Address translation is done using page table which maps virtual page number to physical frame number. Page fault occurs when process accesses a page not currently loaded in RAM causing OS to load it from secondary storage. Demand paging improves memory utilisation since only referenced pages occupy RAM.','avg_human_score':8.25,'quality':'good'},
        {'id':'Q4_A9','answer':'Virtual memory is concept in operating system. Pages are units of virtual memory. Physical memory has frames. When page is not in memory page fault happens. OS loads page from disk.','avg_human_score':3.75,'quality':'below_average'},
        {'id':'Q4_A10','answer':'Virtual memory is technique that creates illusion of large memory using disk storage as backup for RAM. Paging mechanism divides logical memory into fixed size pages and physical memory into same sized frames so any page fits any frame. The page table data structure performs mapping from page numbers to frame numbers enabling virtual to physical address translation. When accessed page is not in physical memory page fault exception occurs and operating system fetches required page from disk to a free frame. Demand paging strategy delays page loading until first access therefore minimising memory consumption.','avg_human_score':8.75,'quality':'excellent'}
      ]
    },
    {
      'id': 'Q5',
      'question': 'Explain the concept of normalisation in databases and its different forms.',
      'max_marks': 10,
      'rubric_points': [
        'normalisation is the process of organising database to reduce redundancy and improve data integrity',
        'first normal form requires atomic values in each column and no repeating groups',
        'second normal form requires no partial dependency where non-key attributes depend on entire primary key',
        'third normal form requires no transitive dependency where non-key attributes depend only on primary key',
        'normalisation reduces update anomalies insertion anomalies and deletion anomalies in the database'
      ],
      'student_answers': [
        {'id':'Q5_A1','answer':'Normalisation is the systematic process of organising a relational database to reduce data redundancy and improve data integrity. First Normal Form requires that each column contains atomic indivisible values and there are no repeating groups or arrays. Second Normal Form builds on 1NF by requiring that every non-key attribute is fully functionally dependent on the entire primary key, thus eliminating partial dependencies that occur in composite key tables. Third Normal Form further requires that no non-key attribute is transitively dependent on the primary key. Normalisation is important because unnormalised tables suffer from update anomalies where changing one fact requires multiple updates, insertion anomalies where some facts cannot be recorded without others, and deletion anomalies where deleting one fact inadvertently removes other facts.','avg_human_score':9.75,'quality':'excellent'},
        {'id':'Q5_A2','answer':'Normalisation reduces redundancy in databases. 1NF requires atomic values and no repeating groups. 2NF removes partial dependencies on primary key. 3NF removes transitive dependencies. Normalisation prevents update insertion and deletion anomalies.','avg_human_score':6.75,'quality':'average'},
        {'id':'Q5_A3','answer':'Normalisation is used in databases. There are different normal forms. 1NF 2NF 3NF are normal forms. They reduce problems in database. Normalisation is important for good database design.','avg_human_score':2.75,'quality':'poor'},
        {'id':'Q5_A4','answer':'Database normalisation is process of structuring relational database to reduce redundancy and data anomalies. First normal form eliminates repeating groups and ensures each attribute contains atomic values. Second normal form eliminates partial dependencies by ensuring every non-primary-key attribute depends on the whole composite primary key hence it only applies when primary key is composite. Third normal form eliminates transitive dependencies where a non-key attribute determines another non-key attribute. Since anomalies cause data inconsistency, normalisation prevents update anomalies, insertion anomalies, and deletion anomalies by ensuring each fact is stored exactly once.','avg_human_score':9.25,'quality':'excellent'},
        {'id':'Q5_A5','answer':'Normalisation is database concept. Normal forms are used. 1NF is first. 2NF is second. 3NF is third. Anomalies are reduced.','avg_human_score':1.75,'quality':'very_poor'},
        {'id':'Q5_A6','answer':'Normalisation organises database tables to minimise redundancy and dependency. 1NF ensures atomic column values with no repeating groups therefore making each row uniquely identifiable. 2NF removes partial dependencies so all non-key columns fully depend on entire primary key which matters when the key is composite. 3NF further eliminates transitive dependencies meaning non-key attributes must depend directly on primary key and not on each other. As a result of normalisation, databases avoid three types of anomalies: update anomalies from redundant data, insertion anomalies from incomplete records, and deletion anomalies from unintended data loss.','avg_human_score':8.75,'quality':'excellent'},
        {'id':'Q5_A7','answer':'Normalisation is for database design. 1NF needs atomic values. 2NF needs full dependency. 3NF needs no transitive dependency. Anomalies are reduced by normalisation.','avg_human_score':4.25,'quality':'below_average'},
        {'id':'Q5_A8','answer':'Normalisation is a technique to organise relational database by reducing redundancy and ensuring data integrity. In first normal form each cell must have single atomic value and there should be no repeating columns. Second normal form requires complete functional dependency on primary key eliminating partial dependencies for composite keys. Third normal form eliminates transitive functional dependencies between non-key attributes. By applying these normal forms we eliminate update anomalies, insertion anomalies, and deletion anomalies.','avg_human_score':8.25,'quality':'good'},
        {'id':'Q5_A9','answer':'Normalisation reduces redundancy. 1NF 2NF 3NF are the forms. Each form has rules. Database becomes better after normalisation. Anomalies are problems in unnormalised database.','avg_human_score':3.25,'quality':'poor'},
        {'id':'Q5_A10','answer':'Normalisation systematically decomposes relations to eliminate redundancy and functional dependency problems. First normal form mandates atomic attribute values and forbids repeating groups. Second normal form addresses partial dependencies in composite key relations ensuring every non-key attribute depends on the complete key. Third normal form requires elimination of transitive dependencies so non-key attributes depend solely on the primary key. These three forms progressively eliminate update anomalies, insertion anomalies, and deletion anomalies.','avg_human_score':9.25,'quality':'excellent'}
      ]
    }
  ]
}

total = sum(len(q['student_answers']) for q in DATASET['questions'])
print(f'✅ Dataset loaded: {len(DATASET["questions"])} questions, {total} student answers')

✅ Dataset loaded: 5 questions, 50 student answers


---
## CELL 6 — Compute 6 metrics for all 50 answers

In [ ]:
# ── CELL 6: Build feature matrix for all 50 answers ───────────────────────
# Takes ~15–20 mins on CPU, ~5 mins on GPU
# Progress bar shows per-answer status

import pandas as pd

rows  = []
total = sum(len(q['student_answers']) for q in DATASET['questions'])
done  = 0

for q in DATASET['questions']:
    for ans in q['student_answers']:
        done += 1
        print(f'  [{done:2d}/{total}] {ans["id"]} ...', end=' ')

        result = evaluate_answer(
            question       = q['question'],
            student_answer = ans['answer'],
            rubric_points  = q['rubric_points'],
            max_marks      = q['max_marks']
        )

        rows.append({
            'answer_id':        ans['id'],
            'question_id':      q['id'],
            'quality':          ans['quality'],
            'human_score':      ans['avg_human_score'],
            'human_norm':       ans['avg_human_score'] / q['max_marks'],
            'max_marks':        q['max_marks'],
            'R': result['R_relevance'],
            'V': result['V_coverage'],
            'C': result['C_consistency'],
            'Q': result['Q_reasoning'],
            'H': result['H_coherence'],
            'IC': result['IC_self_consistency'],
            'default_score': result['final_score']
        })
        print(f'score={result["final_score"]:.1f}  human={ans["avg_human_score"]}')

df = pd.DataFrame(rows)
print(f'\n✅ Feature matrix built: {len(df)} rows x {len(df.columns)} columns')
print('\nMean metric values:')
for col in ['R','V','C','Q','H','IC']:
    print(f'  {col}: {df[col].mean():.3f}')

  [ 1/50] Q1_A1 ... score=6.7  human=9.75
  [ 2/50] Q1_A2 ... score=7.9  human=7.75
  [ 3/50] Q1_A3 ... score=5.9  human=5.25
  [ 4/50] Q1_A4 ... score=4.9  human=3.25
  [ 5/50] Q1_A5 ... score=7.3  human=9.25
  [ 6/50] Q1_A6 ... score=5.2  human=4.0
  [ 7/50] Q1_A7 ... score=6.9  human=7.0
  [ 8/50] Q1_A8 ... score=4.5  human=2.75
  [ 9/50] Q1_A9 ... score=7.1  human=8.25
  [10/50] Q1_A10 ... score=4.9  human=1.75
  [11/50] Q2_A1 ... score=8.0  human=9.75
  [12/50] Q2_A2 ... score=6.8  human=7.75
  [13/50] Q2_A3 ... score=5.3  human=4.75
  [14/50] Q2_A4 ... score=7.2  human=8.75
  [15/50] Q2_A5 ... score=5.2  human=2.25
  [16/50] Q2_A6 ... score=8.0  human=9.0
  [17/50] Q2_A7 ... score=5.3  human=3.0
  [18/50] Q2_A8 ... score=6.3  human=8.25
  [19/50] Q2_A9 ... score=4.6  human=1.75
  [20/50] Q2_A10 ... score=8.0  human=8.25
  [21/50] Q3_A1 ... score=7.2  human=9.75
  [22/50] Q3_A2 ... score=5.6  human=7.25
  [23/50] Q3_A3 ... score=4.2  human=3.25
  [24/50] Q3_A4 ... score=6.6  human

---
## CELL 7 — Weight tuning

In [ ]:
# ── CELL 7: Tune weights (COMPLETE REPLACEMENT) ──────────────────
# Copy everything below and replace your entire Cell 7

from scipy.optimize import minimize, Bounds
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import KFold

# ── predict uses 6 columns: R, V, C, Q, H, IC ────────────────────
def predict(df_subset, weights):
    X = df_subset[['R','V','C','Q','H','IC']].values
    w = np.array([weights['R'], weights['V'], weights['C'],
                  weights['Q'], weights['H'], weights['IC']])
    return np.clip(X @ w, 0, 1)

def get_metrics(y_pred, y_true, max_m):
    pear,  _ = pearsonr(y_pred, y_true)
    spear, _ = spearmanr(y_pred, y_true)
    yp  = np.clip(np.round(y_pred * max_m).astype(int), 0, int(max_m.max()))
    yt  = np.clip(np.round(y_true * max_m).astype(int), 0, int(max_m.max()))
    qwk = cohen_kappa_score(yt, yp, weights='quadratic')
    return {
        'Pearson':  round(float(pear),  4),
        'Spearman': round(float(spear), 4),
        'QWK':      round(float(qwk),   4)
    }

def tune(df_train):
    def obj(w):
        wa = np.abs(w)
        wn = wa / (wa.sum() + 1e-8)
        wt = {'R':wn[0],'V':wn[1],'C':wn[2],
              'Q':wn[3],'H':wn[4],'IC':wn[5]}
        yp   = predict(df_train, wt)
        r, _ = pearsonr(yp, df_train['human_norm'].values)
        return -r

    res = minimize(
        obj,
        x0     = [0.15, 0.30, 0.15, 0.25, 0.10, 0.05],
        method = 'L-BFGS-B',
        bounds = Bounds(
            lb = [0.05, 0.10, 0.05, 0.10, 0.05, 0.02],
            ub = [0.40, 0.60, 0.40, 0.60, 0.30, 0.20]
        ),
        options = {'maxiter': 2000}
    )
    wa = np.abs(res.x)
    wn = wa / wa.sum()
    return {
        'R':  round(float(wn[0]), 4),
        'V':  round(float(wn[1]), 4),
        'C':  round(float(wn[2]), 4),
        'Q':  round(float(wn[3]), 4),
        'H':  round(float(wn[4]), 4),
        'IC': round(float(wn[5]), 4)
    }

# ── Train / test split ────────────────────────────────────────────
split    = int(0.8 * len(df))
df_train = df.iloc[:split].reset_index(drop=True)
df_test  = df.iloc[split:].reset_index(drop=True)

print('Tuning weights on training set...')
tuned_w = tune(df_train)

print('\nOptimal weights:')
for k, v in tuned_w.items():
    print(f'  {k}: {v}')

# ── Evaluate on test set ──────────────────────────────────────────
default_w = {
    'R':  0.15,
    'V':  0.25,   # reduced from 0.30 — coverage alone not enough
    'C':  0.15,
    'Q':  0.30,   # increased from 0.25 — explanation quality matters more
    'H':  0.10,
    'IC': 0.05
}
y_true    = df_test['human_norm'].values
max_m     = df_test['max_marks'].values

m_tuned   = get_metrics(predict(df_test, tuned_w),   y_true, max_m)
m_default = get_metrics(predict(df_test, default_w), y_true, max_m)

print(f'\nTest set results:')
print(f'{"Method":<25} {"Pearson":>9} {"Spearman":>9} {"QWK":>8}')
print('-'*55)
print(f'{"Default weights":<25} {m_default["Pearson"]:>9} '
      f'{m_default["Spearman"]:>9} {m_default["QWK"]:>8}')
print(f'{"Tuned weights":<25} {m_tuned["Pearson"]:>9} '
      f'{m_tuned["Spearman"]:>9} {m_tuned["QWK"]:>8}')

# ── 5-fold cross validation ───────────────────────────────────────
print('\n5-fold cross validation:')
kf     = KFold(n_splits=5, shuffle=True, random_state=42)
cv_qwk = []
for fold, (tr, val) in enumerate(kf.split(df)):
    w  = tune(df.iloc[tr])
    m  = get_metrics(predict(df.iloc[val], w),
                     df.iloc[val]['human_norm'].values,
                     df.iloc[val]['max_marks'].values)
    cv_qwk.append(m['QWK'])
    print(f'  Fold {fold+1}: QWK={m["QWK"]:.4f}  Pearson={m["Pearson"]:.4f}')

print(f'  Mean QWK: {np.mean(cv_qwk):.4f} ± {np.std(cv_qwk):.4f}')
print('\n✅ Weight tuning complete')

Tuning weights on training set...

Optimal weights:
  R: 0.0471
  V: 0.2741
  C: 0.0483
  Q: 0.5646
  H: 0.0471
  IC: 0.0188

Test set results:
Method                      Pearson  Spearman      QWK
-------------------------------------------------------
Default weights              0.8211    0.8328   0.5263
Tuned weights                0.8753    0.8815   0.7538

5-fold cross validation:
  Fold 1: QWK=0.7139  Pearson=0.9300
  Fold 2: QWK=0.7866  Pearson=0.8697
  Fold 3: QWK=0.4316  Pearson=0.9572
  Fold 4: QWK=0.5955  Pearson=0.7654
  Fold 5: QWK=0.5414  Pearson=0.9014
  Mean QWK: 0.6138 ± 0.1254

✅ Weight tuning complete


---
## CELL 8 — Baseline comparison + results table

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# Baseline 1: TF-IDF
tfidf_scores = []
for q in DATASET['questions']:
    ref  = ' '.join(q['rubric_points'])
    docs = [ref] + [a['answer'] for a in q['student_answers']]
    mat  = TfidfVectorizer().fit_transform(docs)
    sims = cos_sim(mat[0:1], mat[1:])[0]
    tfidf_scores.extend(sims.tolist())
y_tfidf = np.array(tfidf_scores)

# Baseline 2: SBERT only
sbert_scores = []
for q in DATASET['questions']:
    ref_emb = sbert.encode(' '.join(q['rubric_points']),
                            convert_to_tensor=True)
    for a in q['student_answers']:
        a_emb = sbert.encode(a['answer'], convert_to_tensor=True)
        sbert_scores.append(float(util.cos_sim(ref_emb, a_emb).clamp(0,1)))
y_sbert = np.array(sbert_scores)

# System scores
y_tuned   = predict(df, tuned_w)
y_default = predict(df, default_w)
y_true_all= df['human_norm'].values
max_m_all = df['max_marks'].values

print('='*62)
print(f'{"Method":<30} {"Pearson":>8} {"Spearman":>9} {"QWK":>7}')
print('='*62)
methods = [
    ('TF-IDF baseline',          y_tfidf),
    ('SBERT-only baseline',      y_sbert),
    ('6-metric default weights', y_default),
    ('6-metric tuned weights',   y_tuned),
]
for name, ypred in methods:
    m = get_metrics(np.clip(ypred, 0, 1), y_true_all, max_m_all)
    print(f'{name:<30} {m["Pearson"]:>8} {m["Spearman"]:>9} {m["QWK"]:>7}')
print('='*62)
print('\n✅ Your 6-metric system should outperform both baselines')

Method                          Pearson  Spearman     QWK
TF-IDF baseline                  0.7582      0.74   0.424
SBERT-only baseline              0.5344    0.5161  0.1572
6-metric default weights         0.8487    0.8068  0.5385
6-metric tuned weights           0.9034    0.8459  0.7638

✅ Your 6-metric system should outperform both baselines


---
## CELL 9 — Error analysis + ablation study

In [ ]:
print('ABLATION STUDY')
print('='*55)
base_qwk = get_metrics(
    predict(df, tuned_w),
    df['human_norm'].values,
    df['max_marks'].values
)['QWK']
print(f'  All 6 metrics (baseline): QWK = {base_qwk:.4f}')
print(f'  {"Remove metric":<20} {"QWK":>8} {"Drop":>8}  Impact')
print('-'*55)

for metric in ['R','V','C','Q','H','IC']:
    w_abl          = tuned_w.copy()
    w_abl[metric]  = 0.0
    rem   = {k: v for k, v in w_abl.items() if v > 0}
    tot   = sum(rem.values())
    if tot > 0:
        for k in rem:
            w_abl[k] = rem[k] / tot
    aq   = get_metrics(
        predict(df, w_abl),
        df['human_norm'].values,
        df['max_marks'].values
    )['QWK']
    drop = base_qwk - aq
    imp  = 'HIGH' if drop > 0.05 else ('MEDIUM' if drop > 0.02 else 'LOW')
    print(f'  Remove {metric:<14} {aq:>8.4f} {drop:>+8.4f}  {imp}')

# ── Top 5 errors ──────────────────────────────────────────────────
df['system_score'] = np.round(predict(df, tuned_w) * df['max_marks'], 2)
df['error']        = (df['system_score'] - df['human_score']).abs()

print(f'\nTOP 5 LARGEST ERRORS')
print('='*55)
for _, row in df.nlargest(5, 'error').iterrows():
    direction = 'over' if row['system_score'] > row['human_score'] else 'under'
    print(f"\n  {row['answer_id']} [{row['quality']}]")
    print(f"  Human: {row['human_score']}  System: {row['system_score']}  "
          f"({direction}scored by {row['error']:.2f})")
    print(f"  R={row['R']:.3f} V={row['V']:.3f} C={row['C']:.3f} "
          f"Q={row['Q']:.3f} H={row['H']:.3f} IC={row['IC']:.3f}")
    if row['V'] < 0.4 and row['human_score'] >= 6:
        print('  → Vocabulary mismatch: SBERT missed concept despite correct answer')
    elif row['Q'] < 0.2 and row['human_score'] >= 7:
        print('  → Implicit reasoning: student used causal language not in marker list')
    elif row['C'] < 0.3 and row['human_score'] >= 6:
        print('  → NLI noise: domain phrasing caused low entailment score')
    else:
        print('  → Combined metric interaction')


ABLATION STUDY
  All 6 metrics (baseline): QWK = 0.7638
  Remove metric             QWK     Drop  Impact
-------------------------------------------------------
  Remove R                0.8063  -0.0425  LOW
  Remove V                0.7723  -0.0085  LOW
  Remove C                0.7889  -0.0251  LOW
  Remove Q                0.3066  +0.4572  HIGH
  Remove H                0.7802  -0.0164  LOW
  Remove IC               0.7897  -0.0259  LOW

TOP 5 LARGEST ERRORS

  Q4_A5 [very_poor]
  Human: 1.75  System: 4.79  (overscored by 3.04)
  R=0.823 V=0.850 C=0.772 Q=0.250 H=0.353 IC=0.650
  → Combined metric interaction

  Q5_A10 [excellent]
  Human: 9.25  System: 6.39  (underscored by 2.86)
  R=0.595 V=0.900 C=0.619 Q=0.520 H=0.540 IC=0.800
  → Combined metric interaction

  Q1_A10 [very_poor]
  Human: 1.75  System: 4.56  (overscored by 2.81)
  R=0.786 V=0.750 C=0.713 Q=0.269 H=0.287 IC=0.750
  → Combined metric interaction

  Q2_A9 [very_poor]
  Human: 1.75  System: 4.53  (overscored by 2.78

---
## CELL 10 — Live evaluation (type your own answer)

In [ ]:
# ── CELL 10: Evaluate any answer interactively ────────────────────────────
# Change the values below and re-run this cell to test any answer

MY_QUESTION = 'Explain deadlock and its four necessary conditions.'

MY_RUBRIC = [
    'deadlock is a situation where processes wait indefinitely for resources',
    'mutual exclusion means only one process can use a resource at a time',
    'hold and wait means a process holds resources while waiting for more',
    'no preemption means resources cannot be forcibly removed',
    'circular wait means a chain of processes each waiting for the next'
]

MY_ANSWER = """
Deadlock is a situation in operating systems where processes cannot proceed
because each is waiting for a resource that another process holds.
The four necessary conditions are mutual exclusion, hold and wait,
no preemption, and circular wait. All four must exist simultaneously.
"""

MAX_MARKS = 10

# ─────────────────────────────────────────────────────────────────────────
result = evaluate_answer(MY_QUESTION, MY_ANSWER, MY_RUBRIC,
                          max_marks=MAX_MARKS, weights=tuned_w)

print('='*55)
print(f'  FINAL SCORE:  {result["final_score"]} / {MAX_MARKS}')
print('='*55)
print(f'  R Relevance   : {result["R_relevance"]}')
print(f'  V Coverage    : {result["V_coverage"]}  '
      f'({result["covered_count"]}/{result["total_points"]} rubric points covered)')
print(f'  C Consistency : {result["C_consistency"]}')
print(f'  Q Reasoning   : {result["Q_reasoning"]}')
print(f'  H Coherence   : {result["H_coherence"]}')
print(f'  IC      : {result["IC_self_consistency"]}')
print('\nRubric point breakdown:')
for pt in result['coverage_detail']:
    icon = '✅' if pt['covered'] else '❌'
    print(f'  {icon} [{pt["similarity"]:.3f}] {pt["rubric_point"]}')

# Feedback
pct = result['final_score'] / MAX_MARKS
if   pct >= 0.8: fb = '🟢 Excellent — well explained with good coverage'
elif pct >= 0.6: fb = '🟡 Good — covers main points, could improve depth'
elif pct >= 0.4: fb = '🟠 Average — several key concepts missing'
else:            fb = '🔴 Insufficient — most key concepts not addressed'
print(f'\nFeedback: {fb}')

  FINAL SCORE:  5.71 / 10
  R Relevance   : 0.8176
  V Coverage    : 0.7  (4/5 rubric points covered)
  C Consistency : 0.5
  Q Reasoning   : 0.58
  H Coherence   : 0.3835
  IC      : 0.6

Rubric point breakdown:
  ✅ [0.894] deadlock is a situation where processes wait indefinitely for resources
  ✅ [0.515] mutual exclusion means only one process can use a resource at a time
  ✅ [0.582] hold and wait means a process holds resources while waiting for more
  ❌ [0.339] no preemption means resources cannot be forcibly removed
  ✅ [0.580] circular wait means a chain of processes each waiting for the next

Feedback: 🟠 Average — several key concepts missing
